### 1. 大模型概述


LLM 是强大的 AI 工具，能够像人类一样理解和生成文本。它们用途广泛，可以撰写内容、翻译语言、进行摘要、回答问题，而无需为每个任务进行专门的训练。

除了文本生成之外，许多模型还支持：
- **工具调用（Tool calling）**——调用外部工具（例如数据库查询或 API 调用），并将结果用于它们的回复中。
- **结构化输出（Structured output）**——模型的回复被约束为遵循某种定义的格式。
- **多模态（Multimodality）**——处理并返回文本之外的数据，例如图像、音频和视频。
- **推理（Reasoning）**——模型执行多步推理以得出结论。

模型是智能体（Agent）的推理引擎。它们驱动智能体的决策过程，决定调用哪些工具、如何解释结果，以及何时给出最终答案。

你所选模型的质量和能力，直接影响到智能体的基线可靠性和性能。不同模型各有所长——有些更擅长遵循复杂指令，有些擅长结构化推理，还有些支持更大的上下文窗口以处理更多信息

### 2. Basic Usage（基本用法）

模型可以通过两种方式使用：
- 配合智能体（With agents）——创建智能体时，可以动态地指定模型。
- 独立使用（Standalone）——可以在智能体循环之外直接调用模型，用于文本生成、分类或信息抽取等任务，而无需智能体框架。

同一个模型接口在两种场景下都能工作，这让你能够灵活地从简单场景开始，并根据需要逐步扩展到更复杂的基于智能体的工作流

#### 2.1 Initialize a model（初始化模型）

在 LangChain 中，独立使用模型最简单的入门方式，是使用 ```init_chat_model```

1. 引入依赖
```
uv add langchain
uv add langchain-deeopseek
```
2. 配置环境
```在系统环境变量中配置 DEEPSEEK_API_KEY=sk-xxx```

In [3]:
from langchain.chat_models import init_chat_model
# 1. 初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-flash")
print(type(model))
response = model.invoke("明天昆山天气怎么样")
print(response.content)

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>
抱歉，我无法实时联网获取天气数据，不能准确告诉你 **2026年5月8日昆山** 的天气情况。

建议你查看：
- 手机自带天气App
- 中国天气网
- 昆山气象台官方发布
- 墨迹天气/彩云天气等

如果你把预报内容发给我，我可以帮你分析穿什么、要不要带伞、出行注意事项。


In [28]:
print(type(response))
# 将AIMessage类型转化成字典
print(type(response.model_dump()))
# 将AIMessage类型转化成json
print(type(response.model_dump_json(indent=2)))

<class 'langchain_core.messages.ai.AIMessage'>
<class 'dict'>
<class 'str'>


```
AIMessage
├── content              最终文本
├── additional_kwargs    附加信息，如 refusal、reasoning_content
├── response_metadata    模型原始元数据，如 token_usage、model_name
├── id                   LangChain 消息 ID
├── tool_calls           工具调用
├── invalid_tool_calls   无效工具调用
└── usage_metadata       标准化 token 用量
```

In [36]:
# 回复内容
print(response.content)

抱歉，我无法实时联网查询天气，没办法准确告诉你明天昆山的具体天气。

建议你打开手机自带天气App，或查看“中国天气网”“墨迹天气”等，搜索 **“昆山 明天天气”** 获取最新预报。

小提示：5月上旬昆山通常比较温暖湿润，可能有阵雨，出门可以备把伞。


In [35]:
# 推理内容
print(response.additional_kwargs.get("reasoning_content"))

我们需要回答用户“明天昆山天气怎么样”。需要查天气。当前日期是2026-05-07，用户问“明天”即2026-05-08。需要昆山天气。我无法直接访问互联网，但可以基于知识？需要实时天气。需要可能使用工具？没有提供工具。作为AI，我可以说明无法实时获取，但可以提供一般建议，或者询问具体日期。但用户期望天气。我们可以尝试根据天气预报知识？当前日期 2026-05-07。我们没有实时数据。需要诚实。也许可以告诉用户我无法联网，建议查看天气应用。但也许需要尽力。可提供昆山明天天气通常？不准确。

需要遵守政策？没有特别。作为API，可能希望回答。可以尝试生成一个模拟天气？但错误信息有风险。应该说不确定，无法获取实时天气。可以提供查询方式。但如果模型有内置实时？没有。

也许可以建议：明天昆山天气请以中国天气网/手机自带天气为准。我可以帮你解读天气。若必须，我可以根据气候平均：5月上旬昆山平均气温约18-26°C，可能有阵雨。但这不叫明天天气。

用户用中文，回答中文。应该简洁。可以表示“我无法实时联网查询，建议打开天气App查看昆山明天预报。5月昆山多雨，建议带伞”。但最好直接承认。

考虑是否要提供“明天（5月8日）昆山”的天气信息？我可以编造吗？不要。

也许可以调用知识：2026年5月8日昆山天气？在训练数据中没有未来。不能。

最终回答：抱歉，我无法实时获取天气数据，不能准确告诉你明天昆山天气。建议查看中国天气网、墨迹天气或手机天气；搜索“昆山 天气 明天”。若需要，我可以帮你根据预报判断穿衣/出行。另：5月上旬昆山通常温暖湿润，可能有雨，出门建议带伞。这样即可。


In [39]:
response.usage_metadata

{'input_tokens': 35,
 'output_tokens': 492,
 'total_tokens': 527,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 415}}

In [40]:
response.tool_calls

[]

#### 2.2 Key methods（关键方法）
- Invoke（调用）——模型接收消息作为输入，在生成完整回复后输出消息。
- Stream（流式）——调用模型，但在输出实时生成的过程中进行流式输出。
- Batch（批量）——以批量的方式向模型发送多个请求，以便更高效地处理。

除了聊天模型之外，LangChain 还支持其他相邻技术，例如嵌入模型（embedding models）和向量存储（vector stores）。

### 3. Parameters（参数）
聊天模型接受可用于配置其行为的参数。受支持的完整参数集合因模型和提供商而异，但标准参数包括：model、api_key、temperature、max_tokens、timeout、max_retries（默认值为 6）

| 参数 | 类型 | 中文含义 | 作用 |
|------|------|---------|------|
| `model` | string（必填） | 模型标识 | 指定用哪个模型，可用 `provider:model` 格式如 `openai:o1` |
| `api_key` | string | API 密钥 | 提供商鉴权，通常用环境变量传入 |
| `temperature` | number | 温度 | 控制随机性：越高越有创意，越低越确定 |
| `max_tokens` | number | 最大 token 数 | 限制回复总 token 数，控制输出长度 |
| `timeout` | number | 超时时间（秒） | 超过则取消请求 |
| `max_retries` | number（默认 6） | 最大重试次数 | 网络超时/限流时自动重发 |

In [ ]:
model = init_chat_model(
    model="deepseek-flash",
    # 传给模型的 kwargs（关键字参数）：
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # 默认值；网络不稳时可调大
)

#### 3.1 Connection resilience（连接韧性）
LangChain 聊天模型会自动以指数退避（exponential backoff）的方式重试失败的 API 请求。默认情况下，对于网络错误、限流（429）和服务器错误（5xx），模型最多重试 6 次。诸如 401（未授权）或 404 之类的客户端错误不会被重试。

对于在不稳定网络上长时间运行的智能体图（agent graphs），请考虑使用更高的 max_retries（例如 10–15），以及一个检查点器（checkpointer），以便在失败时保留进度


📌 注解：
- 指数退避（exponential backoff）：重试不是固定间隔，而是"每次等待时间翻倍"，比如 1 秒 → 2 秒 → 4 秒 → 8 秒，避免把服务器打死。加 jitter（抖动）是在间隔里加随机，防止大量请求同时重试。
- HTTP 状态码速记：
  - 429 = 请求太频繁被限流（rate limit）
  - 5xx = 服务器出错了（服务器的问题）
  - 401 = 没权限/密钥错了（你的问题，重试也没用）
  - 404 = 找不到（重试也没用）
- 关键结论：重试只针对"可能过一会儿就好了"的错误（网络抖动、限流、服务器故障），不重试"重试也没用"的错误（密钥错、地址错）。

In [2]:
model = init_chat_model(
    model="deepseek-flash",
    max_retries=10,  # 网络不稳可调大（默认 6）
    timeout=120,     # 秒；网络慢可调大
)

### 4. Invocation（调用方式详解）

#### 4.1 invoke
- a single message
- a list of messages: 代表对话历史, 每条消息都有 role 来表示这条消息是谁发送的

In [17]:
# 格式一：字典格式
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore développer des applications.


In [4]:
# 格式二：Message对象
# - SystemMessage ↔ {"role": "system", ...}
# - HumanMessage ↔ {"role": "user", ...}
# - AIMessage ↔ {"role": "assistant", ...}
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]
response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore créer des applications.


#### 4.2 stream

In [6]:
for chunk in model.stream("为什么鹦鹉的羽毛色彩斑斓？"):
    print(chunk.text, end="|", flush=True)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

In [10]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]
































































The
The sky
The sky is
The sky is usually
The sky is usually blue
The sky is usually blue during
The sky is usually blue during a
The sky is usually blue during a clear
The sky is usually blue during a clear day
The sky is usually blue during a clear day.
The sky is usually blue during a clear day. At
The sky is usually blue during a clear day. At sunrise
The sky is usually blue during a clear day. At sunrise or
The sky is usually blue during a clear day. At sunrise or sunset
The sky is usually blue during a clear day. At sunrise or sunset it
The sky is usually blue during a clear day. At sunrise or sunset it can
The sky is usually blue during a clear day. At sunrise or sunset it can look
The sky is usually blue during a clear day. At sunrise or sunset it can look orange
The sky is usually blue during a clear day. At sunrise or sunset it can look orange,
The sky is usually blue during a clear day. At sunrise or sunset it ca

#### 4.3 batch

In [13]:
responses = model.batch([
    "为什么鹦鹉的羽毛色彩斑斓",
    "什么是量子计算"
])
for response in responses:
    print(response.content, end="\n")

鹦鹉羽毛色彩斑斓，并不是单一原因造成的，而是**独特色素、结构色、性选择、社交识别和生态适应**共同作用的结果。也不是所有鹦鹉都五彩缤纷，很多种类其实以绿色为主，用来在树冠中伪装。

先从“颜色怎么来”说：  
- **红、黄、橙**：主要来自鹦鹉特有的一类色素——**鹦鹉素（psittacofulvins）**，它们能自己合成，不像很多鸟类那样必须从食物中获取类胡萝卜素。  
- **蓝、绿、紫、紫外、虹彩**：多来自羽毛内部的**结构色**。蓝光由角蛋白和空气的纳米结构散射产生；绿色常常是“黄色鹦鹉素 + 蓝色结构色”叠加的结果。  
- **黑、灰、褐**：主要来自黑色素。

再从进化功能说：  
1. **求偶与性选择**：鲜艳羽毛常是健康和基因质量的信号。羽毛颜色饱满、结构完整，说明个体营养好、抗寄生虫能力强，更容易被配偶选中。  
2. **社交识别**：鹦鹉高度群居、寿命长、社会关系复杂。不同颜色和图案像“身份证”，帮助识别同种、个体、配偶和幼鸟，也能减少不同物种之间的杂交。  
3. **视觉交流**：鹦鹉有高度发达的色觉，许多种类还能看到紫外光。人类看到的颜色只是它们信号系统的一部分，它们眼中的羽毛可能更复杂、更醒目。  
4. **伪装与生态适应**：绿色羽毛让许多鹦鹉融入树叶，躲避猛禽。亮色在热带花果斑驳的环境中也可能打散轮廓，或在飞行、炫耀时传递信息。  
5. **捕食与繁殖的权衡**：太鲜艳容易被捕食者发现，所以自然选择和性选择会拉扯。群居、快速飞行、树冠生活等能在一定程度上降低鲜艳带来的风险。

所以，鹦鹉的斑斓色彩既是它们特殊“调色盘”的产物，也是进化中为了求偶、识别、交流和适应环境而形成的视觉语言。
量子计算是一种**利用量子力学规律来存储、处理和读取信息的计算范式**。它和经典计算机最根本的区别在于：经典计算机用“比特”（bit）表示 0 或 1；量子计算机用量子比特（qubit）表示信息，而量子比特可以处于 0 和 1 的**叠加态**。

一个量子比特可以写成：

\[
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle
\]

其中 \(\alpha\)、\(\beta\) 是复数概率幅，测量时得到 0 的概率是 \(|\alpha|^2\)，得到 1 的概率是 \(|\bet

### 5. Tool calling（工具调用）

### 6. Structured output（结构化输出）